## GenX test

In [1]:
include("GenX.jl")
using .GenX
using HiGHS
using JuMP
using Gurobi
using DataFrames
using YAML

[ Info: Running precompile script for GenX. This may take a few minutes.


  ____           __  __   _ _
 / ___| ___ _ __ \ \/ /  (_) |
| |  _ / _ \ '_ \ \  /   | | |
| |_| |  __/ | | |/  \ _ | | |
 \____|\___|_| |_/_/\_(_)/ |_|
                       |__/
 Version: 0.4.1


┌ Info: Running precompile script for GenX. This may take a few minutes.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\startup\genx_startup.jl:56


  ____           __  __   _ _
 / ___| ___ _ __ \ \/ /  (_) |
| |  _ / _ \ '_ \ \  /   | | |
| |_| |  __/ | | |/  \ _ | | |
 \____|\___|_| |_/_/\_(_)/ |_|
                       |__/
 Version: nothing


In [5]:
case = "..\\example_systems\\TDR_test\\system"
demand_in = GenX.get_demand_dataframe(case);
as_vector(col::Symbol) = collect(skipmissing(demand_in[!, col]))
inputs = Dict()

# Number of time steps (periods)
T = length(as_vector(:Time_Index))
# Number of demand curtailment/lost load segments
SEG = length(as_vector(:Demand_Segment))

## Set indices for internal use
inputs["T"] = T
inputs["SEG"] = SEG
Z = 2

inputs["omega"] = zeros(Float64, T) # weights associated with operational sub-period in the model - sum of weight = 8760
# Weights for each period - assumed same weights for each sub-period within a period
inputs["Weights"] = as_vector(:Sub_Weights) # Weights each period

# Total number of periods and subperiods
inputs["REP_PERIOD"] = convert(Int16, as_vector(:Rep_Periods)[1])
inputs["H"] = convert(Int64, as_vector(:Timesteps_per_Rep_Period)[1])
# Creating sub-period weights from weekly weights
for w in 1:inputs["REP_PERIOD"]
    for h in 1:inputs["H"]
        t = inputs["H"] * (w - 1) + h
        inputs["omega"][t] = inputs["Weights"][w] / inputs["H"]
    end
end


In [6]:
# Create time set steps indicies
inputs["hours_per_subperiod"] = div.(T, inputs["REP_PERIOD"]) # total number of hours per subperiod
hours_per_subperiod = inputs["hours_per_subperiod"] # set value for internal use


168

In [8]:
inputs["START_SUBPERIODS"] = 1:hours_per_subperiod:T # set of indexes for all time periods that start a subperiod (e.g. sample day/week)
inputs["INTERIOR_SUBPERIODS"] = setdiff(1:T, inputs["START_SUBPERIODS"]) # set of indexes for all time periods that do not start a subperiod


501-element Vector{Int64}:
   2
   3
   4
   5
   6
   7
   8
   9
  10
  11
   ⋮
 496
 497
 498
 499
 500
 501
 502
 503
 504

In [19]:
inputs["omega"]

504-element Vector{Float64}:
 26.071428571428573
 26.071428571428573
 26.071428571428573
 26.071428571428573
 26.071428571428573
 26.071428571428573
 26.071428571428573
 26.071428571428573
 26.071428571428573
 26.071428571428573
  ⋮
 13.035714285714286
 13.035714285714286
 13.035714285714286
 13.035714285714286
 13.035714285714286
 13.035714285714286
 13.035714285714286
 13.035714285714286
 13.035714285714286

In [22]:
4380/2/168


13.035714285714286

In [18]:
# Demand in MW for each zone
scale_factor = setup["ParameterScale"] == 1 ? ModelScalingFactor : 1
# Max value of non-served energy
inputs["Voll"] = as_vector(:Voll) / scale_factor # convert from $/MWh $ million/GWh (assuming objective is divided by 1000)
# Demand in MW
inputs["pD"] = extract_matrix_from_dataframe(demand_in,
    DEMAND_COLUMN_PREFIX()[1:(end - 1)],
    prefixseparator = 'z') / scale_factor

UndefVarError: UndefVarError: `setup` not defined

In [11]:
169+168

337

In [15]:
inputs["START_SUBPERIODS"][4]

BoundsError: BoundsError: attempt to access 3-element StepRange{Int64, Int64} at index [4]

In [ ]:
# Genx case_runners file,
case = "..\\example_systems\\15_markets_prm_ro"
settings_path = joinpath(case, "settings")
policies_path = joinpath(case, "policies")
output_folder = joinpath(case, "Results") # Write-output settings YAML file path,
genx_settings = joinpath(settings_path, "genx_settings.yml") # Settings YAML file path,
mysetup = GenX.configure_settings(genx_settings, output_folder) # mysetup dictionary stores settings and GenX-specific parameters,
optimizer = Gurobi.Optimizer
OPTIMIZER =  GenX.configure_solver(settings_path, optimizer)


In [ ]:
myinputs = GenX.load_inputs(mysetup_local, case)

## others

In [ ]:
myinputs =  GenX.load_inputs(mysetup, case)
EP =  GenX.generate_model(mysetup, myinputs, OPTIMIZER)
EP, solve_time =  GenX.solve_model(EP, mysetup)
myinputs["solve_time"] = solve_time
inputs = myinputs
setup = mysetup

In [ ]:
Morris_range = load_dataframe(joinpath(case, "Method_of_morris_range.csv"))
groups = Morris_range[!, :Group]
p_steps = Morris_range[!, :p_steps]
total_num_trajectory = Morris_range[!, :total_num_trajectory][1]
num_trajectory = Morris_range[!, :num_trajectory][1]
len_design_mat = Morris_range[!, :len_design_mat][1]
uncertain_columns = unique(Morris_range[!, :Parameter])
#save_parameters = zeros(length(Morris_range[!,:Parameter]))
gen = inputs["RESOURCES"]
sigma = zeros((1, 2))


In [ ]:
column = uncertain_columns[1]
col_sym = Symbol(lowercase(column))
# column_f is the function to get the value "column" for each generator
column_f = isdefined(GenX, col_sym) ? getfield(GenX, col_sym) :
           r -> getproperty(r, col_sym)

In [ ]:
column_f.(gen) .* (1 .+
               Morris_range[Morris_range[!, :Parameter] .== column, :Lower_bound] ./
               100)

In [ ]:
sigma = [sigma;
             [column_f.(gen) .* (1 .+
               Morris_range[Morris_range[!, :Parameter] .== column, :Lower_bound] ./
               100) column_f.(gen) .*
                    (1 .+
                     Morris_range[Morris_range[!, :Parameter] .== column,
                 :Upper_bound] ./ 100)]]

In [ ]:

for column in uncertain_columns
    col_sym = Symbol(lowercase(column))
    # column_f is the function to get the value "column" for each generator
    column_f = isdefined(GenX, col_sym) ? getfield(GenX, col_sym) :
               r -> getproperty(r, col_sym)
    sigma = [sigma;
             [column_f.(gen) .* (1 .+
               Morris_range[Morris_range[!, :Parameter] .== column, :Lower_bound] ./
               100) column_f.(gen) .*
                    (1 .+
                     Morris_range[Morris_range[!, :Parameter] .== column,
                 :Upper_bound] ./ 100)]]
end

In [ ]:

sigma = sigma[2:end, :]

p_range = mapslices(x -> [x], sigma, dims = 2)[:]

## Test Clustering

In [ ]:
using Clustering

# make a random dataset with 1000 random 5-dimensional points
X = rand(2, 200)

# cluster X into 20 clusters using K-means
R = kmeans(X,5; maxiter=200, display=:iter)

@assert nclusters(R) == 5 # verify the number of clusters

a = assignments(R) # get the assignments of points to clusters
c = counts(R) # get the cluster sizes
M = R.centers # get the cluster centers

## GenX Clustering

In [28]:
include("GenX.jl")
using .GenX
using HiGHS
using JuMP
using Gurobi
using DataFrames
using YAML

# Genx case_runners file,
case = "..\\example_systems\\15_markets_prm_ro"
settings_path = joinpath(case, "settings")
policies_path = joinpath(case, "policies")
output_folder = joinpath(case, "Results") # Write-output settings YAML file path,
genx_settings = joinpath(settings_path, "genx_settings.yml") # Settings YAML file path,
mysetup = GenX.configure_settings(genx_settings, output_folder) # mysetup dictionary stores settings and GenX-specific parameters,
optimizer = Gurobi.Optimizer
OPTIMIZER =  GenX.configure_solver(settings_path, optimizer)
myinputs = GenX.load_inputs(mysetup_local, case)

┌ Info: Running precompile script for GenX. This may take a few minutes.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\startup\genx_startup.jl:56


  ____           __  __   _ _
 / ___| ___ _ __ \ \/ /  (_) |
| |  _ / _ \ '_ \ \  /   | | |
| |_| |  __/ | | |/  \ _ | | |
 \____|\___|_| |_/_/\_(_)/ |_|
                       |__/
 Version: nothing

Configuring Settings
Reading Input CSV Files
Network.csv Successfully Read!
Demand (load) data Successfully Read!
Fuels_data.csv Successfully Read!


┌ Info: Thermal.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:336
┌ Info: Vre.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:336
┌ Info: Storage.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:336
┌ Info: Must_run.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:336
┌ Info: Vre_stor.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:336
┌ Info: Resource_energy_share_requirement.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:727
┌ Info: Resource_capacity_reserve_margin.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:727
┌ Info: Resource_maximum_capacity_require


Summary of resources loaded into the model:
-------------------------------------------------------
	Resource type 		Number of resources
	Thermal        		13
	VRE            		11
	Storage        		16
	Must_run       		7
	VRE_and_storage		4
Total number of resources: 51
-------------------------------------------------------
Generators_variability.csv Successfully Read!
Validating time basis
Maximum_capacity_requirement.csv Successfully Read!
Energy_share_requirement.csv Successfully Read!
CO2_cap.csv Successfully Read!
Planning_reserve_margin.csv Successfully Read!
..\example_systems\15_markets_prm_ro\system\Vre_and_stor_solar_variability.csv Successfully Read!
Zones_markets.csv Successfully Read!
Market_price.csv Successfully Read!
Configuring RO Settings
Dict{Any, Any}("FuelsCost" => 0, "InvestmentCost" => 0, "MarketSellPrices" => 1, "FixedOMCost" => 0, "BudgetOfUncertainty_FixedOMCost" => -1, "UncertaintyBudget" => 0.2, "BudgetOfUncertainty_MarketSellPrices" => -1, "MarketBuyPrices

Dict{Any, Any} with 135 entries:
  "Z"                           => 2
  "VS_ELEC"                     => Int64[]
  "RETROFIT_CAP"                => Int64[]
  "VS_STOR_AC_CHARGE"           => [48, 49, 50, 51]
  "LOSS_LINES"                  => Int64[]
  "dfMaxCO2"                    => [2.0e11;;]
  "STOR_HYDRO_SHORT_DURATION"   => Int64[]
  "RET_CAP_CHARGE"              => Set{Int64}()
  "pC_D_Curtail"                => [5000.0]
  "VS_STOR_DC_DISCHARGE"        => Int64[]
  "NEW_CAP_DC"                  => Int64[]
  "RESOURCE_NAMES_DC_DISCHARGE" => Any[]
  "ZONES_ELEC"                  => Any[]
  "Market_BuyPrices_Delta"      => [0.0 0.0 … 0.0 0.0; 12.4 17.4 … 15.0 26.6]
  "pTrans_Max_Possible"         => [200.0]
  "pNet_Map"                    => [-1.0 1.0]
  "omega"                       => [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0…
  "RET_CAP_ENERGY"              => Int64[]
  "RET_CAP_STOR"                => Int64[]
  ⋮                             => ⋮

In [ ]:
#Remember, when we decide number of hours in interval, the clustering will be for the whole period
# [at least this is my understand]

In [30]:
RESOURCE_ZONES = myinputs["RESOURCE_ZONES"]
ZONES = myinputs["R_ZONES"]

# DEMAND - Demand_data.csv
demand_profiles = [myinputs["pD"][:, l] for l in 1:size(myinputs["pD"], 2)]
demand_col_names = [GenX.DEMAND_COLUMN_PREFIX() * string(l)
                    for l in 1:size(demand_profiles)[1]]
demand_zones = [l for l in 1:size(demand_profiles)[1]]
col_to_zone_map = Dict(demand_col_names .=> 1:length(demand_col_names))

# CAPACITY FACTORS - Generators_variability.csv
solar_profiles = []
wind_profiles = []
var_profiles = []
solar_col_names = []
wind_col_names = []
var_col_names = []
solar_pattern = r"solar|pv"i
wind_pattern = r"wind"i
for r in 1:length(RESOURCE_ZONES)
    if occursin(solar_pattern, RESOURCE_ZONES[r])
        push!(solar_col_names, RESOURCE_ZONES[r])
        push!(solar_profiles, myinputs["pP_Max"][r, :])
    elseif occursin(wind_pattern, RESOURCE_ZONES[r])
        push!(wind_col_names, RESOURCE_ZONES[r])
        push!(wind_profiles, myinputs["pP_Max"][r, :])
    end
    push!(var_col_names, RESOURCE_ZONES[r])
    push!(var_profiles, myinputs["pP_Max"][r, :])
    col_to_zone_map[RESOURCE_ZONES[r]] = ZONES[r]
end

# FUEL - Fuels_data.csv
fuel_col_names = string.(myinputs["fuels"])
fuel_profiles = []
AllFuelsConst = true
for f in 1:length(fuel_col_names)
    push!(fuel_profiles, myinputs["fuel_costs"][fuel_col_names[f]])
    if AllFuelsConst && (minimum(myinputs["fuel_costs"][fuel_col_names[f]]) !=
        maximum(myinputs["fuel_costs"][fuel_col_names[f]]))
        AllFuelsConst = false
    end
end
all_col_names = [demand_col_names; var_col_names; fuel_col_names]
all_profiles = [demand_profiles..., var_profiles..., fuel_profiles...]

59-element Vector{Vector{Float64}}:
 [84.0, 79.0, 74.0, 72.0, 72.0, 76.0, 84.0, 90.0, 97.0, 102.0  …  102.0, 100.0, 100.0, 110.0, 110.0, 108.0, 104.0, 101.0, 95.0, 92.0]
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 

In [35]:
ConstData = []
ConstIdx = []
ConstCols = []
for c in 1:length(all_col_names)
    Const = minimum(all_profiles[c]) == maximum(all_profiles[c])
    if Const
        push!(ConstData, all_profiles[c])
        push!(ConstCols, all_col_names[c])
        push!(ConstIdx, c)
    end
end
all_profiles = [all_profiles[i] for i in 1:length(all_profiles) if i ∉ ConstIdx]
all_col_names = [all_col_names[i] for i in 1:length(all_col_names) if i ∉ ConstIdx]

18-element Vector{String}:
 "Demand_MW_z1"
 "Aquamarine_Westside_z1"
 "Arizona_Solar_z1"
 "Humboldt_Bay_Offshore_Wind_z1"
 "Morro_Bay_Offshore_Wind_z1"
 "New_Mexico_Wind_z1"
 "Northern_California_Solar_z1"
 "Northern_California_Wind_z1"
 "NW_Ext_Tx_Wind_z1"
 "Riverside_Palm_Springs_Wind_z1"
 "Riverside_Solar_z1"
 "Wyoming_Wind_z1"
 "IndianValley_Hydro_z1"
 "GibsonSolar_z1"
 "PutahCreek_z1"
 "Resurgence_Hybrid_z1"
 "WillowSpringsSolar_z1"
 "CA_Natural_Gas"

8760-element Vector{Float64}:
  84.0
  79.0
  74.0
  72.0
  72.0
  76.0
  84.0
  90.0
  97.0
 102.0
   ⋮
 100.0
 100.0
 110.0
 110.0
 108.0
 104.0
 101.0
  95.0
  92.0

In [ ]:
all_profiles